In [1]:
!python tools/export_onnx.py --output-name models/best_ckpt.onnx -f exps/default/yolox_nano_cid.py -c models/best_ckpt.pth


args.decode_in_inference: False
Dummy input shape: torch.Size([1, 320, 320, 3])


2025-07-12 22:31:06.524 | INFO     | __main__:main:88 - args value: Namespace(output_name='models/best_ckpt.onnx', input='images', output='output', opset=11, batch_size=1, dynamic=False, no_onnxsim=False, exp_file='exps/default/yolox_nano_cid.py', experiment_name=None, name=None, ckpt='models/best_ckpt.pth', opts=[], decode_in_inference=False)
d:\git\YOLOX-custom\tools\export_onnx.py:104: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted

In [2]:
import onnx
import numpy as np
import os
import onnx.helper # Import onnx.helper to use tensor_dtype_to_np_dtype

# --- Configuration ---
# Path to your ONNX model file.
# This should be the output from the .pth to ONNX conversion.
onnx_model_path = "models/best_ckpt.onnx"

# --- Pre-checks ---
if not os.path.exists(onnx_model_path):
    print(f"Error: ONNX model file not found at '{onnx_model_path}'")
    print("Please ensure the ONNX conversion was successful and the file exists.")
    exit()

print(f"Inspecting ONNX model: {onnx_model_path}")

try:
    # 1. Load the ONNX model
    model = onnx.load(onnx_model_path)
    graph = model.graph

    # 2. Get input tensor details
    print("\n--- ONNX Input Tensor Details ---")
    for input_node in graph.input:
        print(f"  Name: {input_node.name}")
        # Get shape from input_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in input_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        # Use onnx.helper.tensor_dtype_to_np_dtype to avoid DeprecationWarning
        print(f"  Dtype: {onnx.helper.tensor_dtype_to_np_dtype(input_node.type.tensor_type.elem_type)}")
        print("-" * 30)

    # 3. Get output tensor details
    print("\n--- ONNX Output Tensor Details ---")
    for output_node in graph.output:
        print(f"  Name: {output_node.name}")
        # Get shape from output_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in output_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        # Use onnx.helper.tensor_dtype_to_np_dtype to avoid DeprecationWarning
        print(f"  Dtype: {onnx.helper.tensor_dtype_to_np_dtype(output_node.type.tensor_type.elem_type)}")
        print("-" * 30)

    print("\nONNX model inspection complete.")

except Exception as e:
    print(f"\nAn error occurred during ONNX model inspection: {e}")
    print("Please ensure:")
    print("1. The ONNX model file at '{onnx_model_path}' is valid.")
    print("2. You have `onnx` installed (`pip install onnx`).")



Inspecting ONNX model: models/best_ckpt.onnx

--- ONNX Input Tensor Details ---
  Name: images
  Shape: [1, 320, 320, 3]
  Dtype: float32
------------------------------

--- ONNX Output Tensor Details ---
  Name: output
  Shape: [1, 2100, 7]
  Dtype: float32
------------------------------

ONNX model inspection complete.


In [ ]:
import onnx
import tensorflow as tf
import onnx_tf

def convert_onnx_to_tflite(input_onnx_model, output_tflite_model):
    # Step 1: Convert ONNX to TensorFlow SavedModel
    tf_model_path = input_onnx_model.replace(".onnx", "_tf")
    try:
        onnx_model = onnx.load(input_onnx_model)
        tf_rep = onnx_tf.backend.prepare(onnx_model)
        tf_rep.export_graph(tf_model_path)
        print(f"Successfully converted ONNX to TensorFlow SavedModel at: {tf_model_path}")
    except Exception as e:
        print(f"Error converting ONNX to TensorFlow SavedModel: {e}")
        return
 
    # Step 2: Convert TensorFlow SavedModel to TFLite
    try:
        converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
        tflite_model = converter.convert()
        with open(output_tflite_model, 'wb') as f:
            f.write(tflite_model)
        print(f"Successfully converted TensorFlow SavedModel to TFLite at: {output_tflite_model}")
    except Exception as e:
        print(f"Error converting TensorFlow SavedModel to TFLite: {e}")

# Example usage with your provided paths
input_onnx_model = "models/best_ckpt.onnx"
output_tflite_model = "models/best_ckpt.tflite"
convert_onnx_to_tflite(input_onnx_model, output_tflite_model)

c:\Users\Wave\.conda\envs\yolox-tf\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\Wave\.conda\envs\yolox-tf\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.11.0 and strictly below 2.14.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.15.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you 

Instructions for updating:
Use `tf.image.resize(...method=ResizeMethod.NEAREST_NEIGHBOR...)` instead.


INFO:absl:Function `__call__` contains input name(s) x, y with unsupported characters which will be renamed to transpose_343_x, add_94_y in the SavedModel.
INFO:absl:Found untraced functions such as gen_tensor_dict while saving (showing 1 of 1). These functions will not be directly callable after loading.


INFO:tensorflow:Assets written to: models/best_ckpt_tf\assets


INFO:tensorflow:Assets written to: models/best_ckpt_tf\assets
INFO:absl:Writing fingerprint to models/best_ckpt_tf\fingerprint.pb


Successfully converted ONNX to TensorFlow SavedModel at: models/best_ckpt_tf
Successfully converted TensorFlow SavedModel to TFLite at: models/best_ckpt.tflite


In [4]:
import tensorflow as tf

tflite_model_path = "models/best_ckpt.tflite"

try:
    # Load the TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print("\n--- Input Details ---")
    for i, detail in enumerate(input_details):
        print(f"Input {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")

    print("\n--- Output Details ---")
    for i, detail in enumerate(output_details):
        print(f"Output {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")

except Exception as e:
    print(f"Error inspecting TFLite model: {e}")


--- Input Details ---
Input 0:
  Name: serving_default_images:0
  Shape: [  1 320 320   3]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)

--- Output Details ---
Output 0:
  Name: PartitionedCall:0
  Shape: [   1 2100    7]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
